# Task 1.3 — Baseline and Improvement

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE TPAMI

---

## 1. Baseline Method the Paper Compares Against

The primary baseline is the **rigid HOG-based linear SVM detector** introduced by Dalal and Triggs (2005). This detector uses a single HOG template scanned across a feature pyramid, trained with a standard linear SVM on positive and negative windows.

In the paper's terminology, this corresponds to the **root-only model** — a single-component model with no part filters and no deformation. The paper explicitly evaluates this as an ablation in Table 3 of Section 6, showing the root filter alone (similar to Dalal & Triggs) as a comparison point.

Additional baselines referenced include:
- **Dalal & Triggs (CVPR 2005)** — the original HOG + linear SVM person detector.
- **Histogram intersection kernel SVM** (Maji et al., 2008) — a nonlinear kernel method on HOG.
- Previous PASCAL VOC competition entries using bag-of-words and spatial pyramid methods.

**Paper Reference:** Section 6 (Experimental Results), Table 3, Table 4.

## 2. Limitation of the Baseline

The rigid HOG template detector has several critical limitations:

**a) No handling of deformation or articulation.**  
A single rigid template must capture *all* variations of an object category's appearance in one fixed-size weight vector. For a category like "person," different body poses (standing, sitting, walking, arms raised) produce dramatically different HOG patterns. A single rigid template can only learn a blurred average, which matches none of the specific poses well.

**b) Single scale, single viewpoint.**  
The rigid detector does not model that objects look different from different viewpoints. A side-view car and a frontal-view car have very different gradient patterns, and a single template cannot represent both effectively.

**c) Lack of fine-grained detail.**  
The root filter operates at a relatively coarse spatial resolution (8×8 pixel cells). Fine details like facial features or wheel spokes are lost. Without higher-resolution part filters, the system cannot leverage fine-grained appearance cues.

As a result, the rigid detector achieves **significantly lower Average Precision (AP)** on the PASCAL VOC benchmark compared to the full DPM system.

**Paper Reference:** Section 1 (Introduction) discusses limitations of rigid templates; Section 6, Table 3 quantifies the performance gap.

## 3. How the Proposed Method Solves This Limitation

The DPM addresses each limitation of the rigid baseline:

**a) Deformable parts handle articulation.**  
By adding part filters connected to the root via deformable springs (quadratic displacement cost), the model can accommodate variations in part positions. For example, a person's arm can be at their side or raised, and the arm's part filter will shift accordingly — the deformation penalty ensures this shift is controlled but flexible. (Section 3, Equation 3)

**b) Mixture components handle viewpoint.**  
The model uses a **mixture of star models** — multiple components, each with its own root and part filters. Different components naturally specialize to different viewpoints or aspect ratios during training. For example, the car model might learn separate components for side, frontal, and rear views. (Section 3.2)

**c) Part filters provide high-resolution detail.**  
Part filters operate at **twice the spatial resolution** of the root filter (4×4 pixel cells vs. 8×8). This allows the model to capture fine-grained appearance cues for discriminative object parts while the root provides global context. (Section 2.1, Figure 4)

**d) Latent SVM enables end-to-end discriminative training.**  
Rather than requiring part annotations, the system learns part positions as latent variables during discriminative training. This means the model discovers the most useful parts automatically. (Section 4)

**Quantitative improvement:** On PASCAL VOC 2007, the full DPM system achieves a mean AP of ~28% compared to ~22% for the root-only model (Table 3) — a relative improvement of ~27%.

**Paper Reference:** Section 3 (Model), Section 4 (Training), Section 6 Table 3 (Ablation results).

## 4. Scenario Where the Proposed Method Might Fail

Despite its improvements, the DPM can fail in several scenarios:

**Primary failure scenario: Objects without consistent part structure.**

The DPM assumes objects have a roughly consistent set of parts with predictable spatial relationships to a root. Consider detecting **amorphous or highly deformable objects** such as:

- **Blankets or cloths** draped in arbitrary shapes — no consistent parts or spatial layout.
- **Liquids or smoke** — no rigid structure at all.
- **Flocks of birds or herds of animals** — the group has no consistent internal structure.

In these cases, the part-based model offers no advantage over a rigid template, and may even perform worse because the additional model complexity (more parameters) increases the risk of overfitting.

**Additional failure scenarios:**

- **Heavy occlusion:** If the root region or multiple parts are occluded, the model's score drops significantly. The DPM does not have an explicit occlusion handling mechanism (Section 5.1 mentions bounding box prediction as a partial remedy, but not robust occlusion modeling).
  
- **Small objects:** At very small scales, the feature pyramid has too few HOG cells to form meaningful root or part responses. The paper acknowledges difficulty with small objects in the VOC evaluation.

- **Visually similar categories:** HOG features plus linear filters may not have enough discriminative power to separate categories with similar shapes but different semantics (e.g., "chair" vs. "sofa").

**Paper Reference:** Section 6.1 discusses per-category results revealing categories where DPM struggles; Figure 8 shows detection examples including failures.

---

## Summary Table

| Aspect | Details |
|--------|--------|
| **Baseline** | Rigid HOG + linear SVM detector (Dalal & Triggs, 2005) |
| **Baseline Limitation** | Cannot handle articulation, viewpoint variation, or capture fine detail |
| **DPM Solution** | Deformable parts + mixture components + high-res part filters + Latent SVM |
| **DPM Failure Mode** | Objects without consistent part structure; heavy occlusion; small objects |